# 🚀 MedSecure - Local Training

Questo notebook ti permette di eseguire **MedSecure** in locale sul tuo computer.

## ⚙️ Requisiti

- Python 3.8+
- GPU con CUDA (consigliato) o CPU
- Dipendenze installate da `requirements.txt`

## 📊 Tempi Stimati (GPU)

- Setup: 2-5 min
- Train Victim Model: 15-20 min
- Train RL Agent: 2-3 ore
- **Totale: ~3-4 ore**

**Nota**: I tempi possono variare in base all'hardware.

## 1️⃣ Verifica Hardware

In [20]:
# Verifica GPU/CPU disponibile
import torch
import platform

print(f"🖥️  Sistema: {platform.system()} {platform.machine()}")
print(f"🐍 Python: {platform.python_version()}")
print(f"🔥 PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    print(f"\n🎮 CUDA disponibile: True")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f"\n🍎 Apple MPS disponibile: True")
    DEVICE = 'mps'
else:
    print("\n⚠️  Nessuna GPU disponibile, usando CPU (sarà più lento)")
    DEVICE = 'cpu'

print(f"\n✅ Device selezionato: {DEVICE}")

🖥️  Sistema: Linux x86_64
🐍 Python: 3.12.3
🔥 PyTorch: 2.10.0+cu128

🎮 CUDA disponibile: True
   GPU: NVIDIA GeForce RTX 4090
   Memoria: 25.3 GB

✅ Device selezionato: cuda


## 2️⃣ Setup Progetto

In [2]:
import sys
print(sys.executable)


/home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/bin/python


In [3]:
# Configura ambiente locale
import os
import sys
from pathlib import Path

# Trova la directory del progetto (dove si trova questo notebook)
PROJECT_DIR = Path(os.getcwd()).resolve()

# Se il notebook è in una sottodirectory, cerca la root del progetto
if not (PROJECT_DIR / 'setup.py').exists():
    # Cerca setup.py nelle directory parent
    for parent in PROJECT_DIR.parents:
        if (parent / 'setup.py').exists():
            PROJECT_DIR = parent
            break

os.chdir(PROJECT_DIR)
print(f"📂 Directory progetto: {PROJECT_DIR}")

# Aggiungi al Python path
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("✅ Python path configurato")

📂 Directory progetto: /home/demetra/Desktop/Manuel/github-repo/MedSecure
✅ Python path configurato


## 3️⃣ Verifica Dipendenze

**Nota**: Se non hai ancora installato le dipendenze, esegui prima:
```bash
pip install -r requirements.txt
```

In [5]:
# Verifica dipendenze
import importlib

dependencies = [
    ('torch', 'PyTorch'),
    ('torchvision', 'TorchVision'),
    ('medmnist', 'MedMNIST'),
    ('gymnasium', 'Gymnasium'),
    ('stable_baselines3', 'Stable-Baselines3'),
    ('cv2', 'OpenCV'),
    ('PIL', 'Pillow'),
    ('lpips', 'LPIPS'),
    ('yaml', 'PyYAML'),
    ('matplotlib', 'Matplotlib'),
]

missing = []
for module, name in dependencies:
    try:
        importlib.import_module(module)
        print(f"✅ {name}")
    except ImportError:
        print(f"❌ {name} - non installato")
        missing.append(name)

if missing:
    print(f"\n⚠️  Dipendenze mancanti: {', '.join(missing)}")
    print("   Esegui: pip install -r requirements.txt")
else:
    print("\n✅ Tutte le dipendenze sono installate!")

✅ PyTorch
✅ TorchVision
✅ MedMNIST
✅ Gymnasium
✅ Stable-Baselines3
✅ OpenCV
✅ Pillow
✅ LPIPS
✅ PyYAML
✅ Matplotlib

✅ Tutte le dipendenze sono installate!


In [6]:
# Verifica moduli locali del progetto
local_modules = ['data', 'models', 'attacks', 'rl', 'evaluation']

print("📦 Verifico moduli locali...\n")
for module in local_modules:
    try:
        importlib.import_module(module)
        print(f"✅ {module}")
    except ImportError as e:
        print(f"❌ {module}: {e}")

print("\n🎉 Setup completato!")

📦 Verifico moduli locali...

✅ data
✅ models
✅ attacks
✅ rl
✅ evaluation

🎉 Setup completato!


## 4️⃣ Download Dataset

In [7]:
# Download MedMNIST dataset
from data.datasets import download_medmnist

print("📥 Downloading PathMNIST dataset...")
download_medmnist(datasets=['pathmnist'])
print("✅ Dataset scaricato!")

📥 Downloading PathMNIST dataset...


100.0%


  ✓ pathmnist downloaded
All datasets downloaded successfully!
✅ Dataset scaricato!


## 5️⃣ Train Victim Model

Addestriamo un modello ResNet50 da attaccare (15-20 min con GPU).

In [ ]:
# Train SimpleCNN victim model
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/train_victim.py',
    '--model', 'simplecnn',
    '--dataset', 'pathmnist',
    '--epochs', '30',
    '--batch-size', '128',
    '--lr', '0.001',
    '--native-resolution',  # usa 28x28 nativo
    '--device', DEVICE,
    '--checkpoint-dir', 'checkpoints/victim'
]

print(f"🚀 Training SimpleCNN...")
result = subprocess.run(cmd)
print("✅ SimpleCNN trained!" if result.returncode == 0 else f"❌ Errore: {result.returncode}")

In [54]:
# Train ResNet18 victim model
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/train_victim.py',
    '--model', 'resnet18',
    '--dataset', 'pathmnist',
    '--epochs', '20',
    '--native',
    '--checkpoint-dir', 'checkpoints/victim'
]

print(f"🚀 Training ResNet18...")
result = subprocess.run(cmd)
print("✅ ResNet18 trained!" if result.returncode == 0 else f"❌ Errore: {result.returncode}")

🚀 Training ResNet18...


2026-02-08 11:02:52,476 - __main__ - INFO - Training resnet18 on pathmnist
2026-02-08 11:02:52,476 - __main__ - INFO - Configuration: epochs=20, batch_size=64, lr=0.001
2026-02-08 11:02:52,476 - __main__ - INFO - Using native 28×28 resolution (no resize, no pretrained preprocessing)
2026-02-08 11:03:06,425 - __main__ - INFO - Saved checkpoint to checkpoints/victim/resnet18_pathmnist_best.pth
2026-02-08 11:03:06,426 - __main__ - INFO - Epoch 1/20 | Train Loss: 0.5873 | Train Acc: 78.97% | Val Loss: 0.3298 | Val Acc: 88.04% | Best: 88.04%
2026-02-08 11:03:17,582 - __main__ - INFO - Saved checkpoint to checkpoints/victim/resnet18_pathmnist_best.pth
2026-02-08 11:03:17,582 - __main__ - INFO - Epoch 2/20 | Train Loss: 0.2928 | Train Acc: 89.72% | Val Loss: 0.2162 | Val Acc: 92.36% | Best: 92.36%
2026-02-08 11:03:28,571 - __main__ - INFO - Epoch 3/20 | Train Loss: 0.2198 | Train Acc: 92.28% | Val Loss: 0.2297 | Val Acc: 91.23% | Best: 92.36%
2026-02-08 11:03:39,770 - __main__ - INFO - Saved 

✅ ResNet18 trained!


In [17]:
# Train victim model
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/train_victim.py',
    '--model', 'resnet50',
    '--dataset', 'pathmnist',
    '--epochs', '10',
    '--batch-size', '64',
    '--lr', '0.001',
    '--checkpoint-dir', 'checkpoints/victim'
]

print(f"🚀 Comando: {' '.join(cmd)}\n")
result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n✅ Victim model trained!")
else:
    print(f"\n❌ Errore durante il training (exit code: {result.returncode})")

🚀 Comando: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/bin/python experiments/train_victim.py --model resnet50 --dataset pathmnist --epochs 10 --batch-size 64 --lr 0.001 --checkpoint-dir checkpoints/victim



2026-02-03 17:41:29,176 - __main__ - INFO - Training resnet50 on pathmnist
2026-02-03 17:41:29,176 - __main__ - INFO - Configuration: epochs=10, batch_size=64, lr=0.001
2026-02-03 17:43:36,463 - __main__ - INFO - Saved checkpoint to checkpoints/victim/resnet50_pathmnist_best.pth
2026-02-03 17:43:36,463 - __main__ - INFO - Epoch 1/10 | Train Loss: 0.2123 | Train Acc: 92.88% | Val Loss: 0.2776 | Val Acc: 90.62% | Best: 90.62%
2026-02-03 17:45:40,674 - __main__ - INFO - Saved checkpoint to checkpoints/victim/resnet50_pathmnist_best.pth
2026-02-03 17:45:40,674 - __main__ - INFO - Epoch 2/10 | Train Loss: 0.1137 | Train Acc: 96.18% | Val Loss: 0.1319 | Val Acc: 95.46% | Best: 95.46%
2026-02-03 17:47:44,759 - __main__ - INFO - Epoch 3/10 | Train Loss: 0.0861 | Train Acc: 97.09% | Val Loss: 0.1642 | Val Acc: 94.99% | Best: 95.46%
2026-02-03 17:49:48,983 - __main__ - INFO - Saved checkpoint to checkpoints/victim/resnet50_pathmnist_best.pth
2026-02-03 17:49:48,983 - __main__ - INFO - Epoch 4/10

KeyboardInterrupt: 

In [26]:
import sys
import subprocess

# Installa numpy
subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])

# Verifica
import numpy as np
print(f"✅ Numpy installato: versione {np.__version__}")

✅ Numpy installato: versione 2.4.2


In [8]:
# Verifica test accuracy del modello sul dataset
import torch
from pathlib import Path
from tqdm import tqdm
import sys
import numpy

# Add project paths
sys.path.insert(0, str(Path.cwd()))

from data import get_dataset, get_dataloader
from models import get_victim_model

# Configuration
checkpoint_path = Path('checkpoints/victim/resnet18_pathmnist_best.pth')
dataset_name = 'pathmnist'
batch_size = 32
n_samples = None  # None = intero test set, oppure es. 1024 per subset

if checkpoint_path.exists():
    # Load model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    print(f"📊 Loading Victim Model...")
    print(f"   Model: {checkpoint.get('model_name', 'unknown')}")
    print(f"   Checkpoint Accuracy: {checkpoint.get('accuracy', 0):.2%}")
    print(f"   Epoch: {checkpoint.get('epoch', 0)}")
    
    # Initialize model
    model = get_victim_model(
        checkpoint['model_name'],
        num_classes=checkpoint['n_classes']
    )
    model.load_state_dict(checkpoint['state_dict'])
    model = model.to(device)
    model.eval()
    
    # Load test dataset
    dataset = get_dataset(dataset_name, split='test')
    loader = get_dataloader(
        dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=0  # Disable multiprocessing
    )
    
    # Calculate test accuracy
    correct = 0
    total = 0
    
    print(f"\n🔍 Calculating Test Accuracy...")
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            if n_samples is not None and total >= n_samples:
                break
            
            images = images.to(device)
            labels = labels.to(device).squeeze()
            
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    test_accuracy = correct / total
    
    print("\n" + "="*60)
    print(f"✅ ACTUAL Test Accuracy: {test_accuracy:.2%} ({correct}/{total})")
    print("="*60)
    
    # Confronto con checkpoint
    checkpoint_acc = checkpoint.get('accuracy', 0)
    diff = test_accuracy - checkpoint_acc
    print(f"\nDifference from checkpoint: {diff:+.2%}")
    
else:
    print(f"❌ Checkpoint non trovato: {checkpoint_path}")

📊 Loading Victim Model...
   Model: resnet18
   Checkpoint Accuracy: 93.42%
   Epoch: 0

🔍 Calculating Test Accuracy...


Evaluating: 100%|██████████| 225/225 [00:11<00:00, 19.53it/s]


✅ ACTUAL Test Accuracy: 91.71% (6585/7180)

Difference from checkpoint: -1.71%


## 6️⃣ Baseline Attacks (Opzionale)

Test attacchi tradizionali per confronto (3-5 min).

In [60]:
# Baseline attacks - CORRETTO
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/run_baselines.py',
    '--model-path', 'checkpoints/victim/resnet18_pathmnist_best.pth',
    '--dataset', 'pathmnist',
    '--epsilon', '0.03',
    '--output-dir', 'results/baselines',
    '--verify-accuracy'
]

print(f"🚀 Running baseline attacks...")
result = subprocess.run(cmd)
print("✅ Done!" if result.returncode == 0 else f"❌ Errore: {result.returncode}")


🚀 Running baseline attacks...


2026-02-08 11:29:52,023 - __main__ - INFO - Loaded victim model: resnet18
2026-02-08 11:29:52,023 - __main__ - INFO - Model accuracy: 0.9943022790883647
2026-02-08 11:29:52,023 - __main__ - INFO - Native mode: True (28x28)
2026-02-08 11:29:52,023 - __main__ - INFO - Normalization stats: mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]



MODEL ACCURACY VERIFICATION


Verifying accuracy:  14%|█▍        | 32/225 [00:00<00:02, 91.91it/s]
2026-02-08 11:29:52,662 - __main__ - INFO - Test Accuracy: 92.77% (950/1024)
2026-02-08 11:29:52,663 - __main__ - INFO - Running FGSM attack...



Test Accuracy: 92.77%

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


...

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth


Running bim:  14%|█▍        | 32/225 [00:05<00:31,  6.16it/s]
2026-02-08 11:30:05,569 - __main__ - INFO - BIM Results:
2026-02-08 11:30:05,570 - __main__ - INFO -   ASR: 91.50%
2026-02-08 11:30:05,570 - __main__ - INFO -   SSIM: 0.9423
2026-02-08 11:30:05,570 - __main__ - INFO -   PSNR: 32.55 dB
2026-02-08 11:30:05,570 - __main__ - INFO - Running CW attack...


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth


Running cw:   5%|▌         | 12/225 [00:30<07:52,  2.22s/it]

KeyboardInterrupt: 

## 7️⃣ Train RL Agent

**⚠️ IMPORTANTE**: Questo richiederà 2-3 ore con GPU.

**Tips**:
- Puoi interrompere e riprendere (i checkpoint vengono salvati)
- Monitora con TensorBoard (cella successiva)

In [11]:

import subprocess
import sys

cmd = [
    sys.executable, 'experiments/train_agent.py',
    '--model-path', 'checkpoints/victim/resnet18_pathmnist_best.pth',
    '--dataset', 'pathmnist',
    '--timesteps', '100000',
    '--batch-size', '256',
    '--output-dir', 'results/agent_curriculum_PPO',
    '--algorithm', 'ppo'
]

print(f"🚀 Comando: {' '.join(cmd)}\n")
result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n✅ RL Agent trained!")
else:
    print(f"\n❌ Errore (exit code: {result.returncode})")

🚀 Comando: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/bin/python experiments/train_agent.py --model-path checkpoints/victim/resnet18_pathmnist_best.pth --dataset pathmnist --timesteps 100000 --batch-size 256 --output-dir results/agent_curriculum_PPO --algorithm ppo



2026-02-07 13:51:40,928 - __main__ - INFO - Loading dataset: pathmnist
2026-02-07 13:51:42,002 - __main__ - INFO - Initializing environment...
2026-02-07 13:51:42,008 - __main__ - INFO - Initializing reward function...
/home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Using PPO (better for curriculum learning, no replay buffer)

Starting training for 100,000 timesteps
Log directory: results/agent_curriculum_PPO/logs



2026-02-07 13:51:42,316 - __main__ - INFO -   Using multi-objective reward function with LPIPS
2026-02-07 13:51:42,316 - __main__ - INFO - Curriculum Phase 0: easy
2026-02-07 13:51:42,316 - __main__ - INFO -   Epsilon range: (0.1, 0.3)
2026-02-07 13:51:42,316 - __main__ - INFO -   Allowed strategies: ['pixel']
2026-02-07 13:51:42,316 - __main__ - INFO - Initializing PPO agent...
/home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Logging to results/agent_curriculum_PPO/logs/tensorboard/PPO_0

TRAINING STARTED - Applying initial curriculum

🎯 Curriculum Phase 0: easy
   Epsilon: (0.1, 0.3)
   Strategies: ['pixel']
   1% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 995/100,000  [ 0:00:34 < 0:56:43 , 29 it/s ]
[Step 1000] [Phase 0: easy] Success Rate: 88.00%m 995/100,000  [ 0:00:34 < 0:56:43 , 29 it/s ]
  [Strategy ASR] PIXEL: 96.0%━━━━━━━━━━━━━━━━ 995/100,000  [ 0:00:34 < 0:56:43 , 29 it/s ]
---------------------------------━━━━━━━━━━━━━━ 1,022/100,000  [ 0:00:35 < 0:56:58 , 29 it/s ]
| rollout/           |          |
|    ep_len_mean     | 1.12     |
|    ep_rew_mean     | 21.9     |
| time/              |          |
|    fps             | 28       |
|    iterations      | 1        |
|    time_elapsed    | 35       |
|    total_timesteps | 1024     |
---------------------------------
   2% ╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 1,996/100,000  [ 0:01:09 < 0:56:34 , 29 it/s ]
[Step 2000] [Phase 0: easy] Success Rate: 92.00%━━━━ 1,996/100,0

In [61]:
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/train_hierarchical.py',
    '--model-path', 'checkpoints/victim/resnet18_pathmnist_best.pth',
    '--dataset', 'pathmnist',
    '--timesteps', '100000',
    '--output-dir', 'results/hierarchical_agent_3',
    '--curriculum'
]

print(f"🚀 Comando: {' '.join(cmd)}\n")
result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n✅ RL Agent trained!")
else:
    print(f"\n❌ Errore (exit code: {result.returncode})")

🚀 Comando: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/bin/python experiments/train_hierarchical.py --model-path checkpoints/victim/resnet18_pathmnist_best.pth --dataset pathmnist --timesteps 100000 --output-dir results/hierarchical_agent_3 --curriculum


  HIERARCHICAL RL AGENT TRAINING
  Dataset:       pathmnist (9 classes)
  Architecture:  resnet18
  Device:        cuda
  Timesteps:     100,000
  Curriculum:    Enabled
  Output:        results/hierarchical_agent_3

[1/4] Loading victim model...
      ✓ Loaded resnet18 from checkpoints/victim/resnet18_pathmnist_best.pth
      ✓ Native mode: 28x28 images
      ✓ Normalization: mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]
[2/4] Loading dataset...
      ✓ Loaded pathmnist (89996 samples)
[3/4] Creating environment...
      ✓ Environment ready
[4/4] Creating hierarchical agent...
      ✓ Agent ready (Meta-Controller: DQN, Controller: PPO)
      ✓ Initial strategies: ['PIXEL']
      ✓ Initial epsilon budget: 0.1

  Curriculum Sc

Training:  10%|██▊                         | 10000/100000 [15:43<2:23:09] , ph=1, ep=3335, rew=19.9, sr=77%, bgt=0.10, e_a=0.048, str=P:100%


  Phase 2: + FREQUENCY
  Strategies: ['PIXEL', 'FREQUENCY']
  Epsilon budget: 0.06



Training:  25%|█████▊                 | 25003/100000 [36:51<1:08:51] , ph=2, ep=9048, rew=19.6, sr=77%, bgt=0.06, e_a=0.035, str=P:74% F:26%

Agent saved to results/hierarchical_agent_3/checkpoint_25000
  💾 Checkpoint saved: results/hierarchical_agent_3/checkpoint_25000


...


  Phase 4: Expert
  Strategies: ['PIXEL', 'FREQUENCY', 'SEMANTIC']
  Epsilon budget: 0.02



Training:  75%|██████████▌   | 75001/100000 [2:14:56<1:27:00] , ph=4, ep=24356, rew=10.7, sr=35%, bgt=0.02, e_a=0.009, str=P:38% F:39% S:23%

Agent saved to results/hierarchical_agent_3/checkpoint_75000
  💾 Checkpoint saved: results/hierarchical_agent_3/checkpoint_75000


Training: 100%|███████████████| 100000/100000 [3:06:36<00:00] , ph=4, ep=31569, rew=16.8, sr=61%, bgt=0.02, e_a=0.011, str=P:35% F:39% S:26%



  SAVING RESULTS
Agent saved to results/hierarchical_agent_3
  ✓ Agent saved to results/hierarchical_agent_3
  ✓ Stats saved to results/hierarchical_agent_3/training_stats.json

  TRAINING COMPLETE
  Total Episodes:    31,602
  Final Success Rate: 65.4%
  Final Mean Reward:  17.48
  Strategy Usage:
    • PIXEL: 11,134 (35.2%)
    • FREQUENCY: 12,373 (39.2%)
    • SEMANTIC: 8,095 (25.6%)


✅ RL Agent trained!


In [ ]:
# Monitora training con TensorBoard
# Esegui in un terminale separato:
# tensorboard --logdir results/agent/logs

# Oppure usa l'estensione Jupyter:
%load_ext tensorboard
%tensorboard --logdir results/agent/logs

## 8️⃣ Evaluation

In [65]:
# Evaluate trained agent
import subprocess
import sys

cmd = [
    sys.executable, 'experiments/evaluate.py',
    '--agent', 'results/hierarchical_agent_3/hierarchical_agent.pt',
    '--victim-model', 'checkpoints/victim/resnet18_pathmnist_best.pth',
    '--dataset', 'pathmnist',
    '--n-samples', '500',
    '--output-dir', 'results/evaluation',
    '--epsilon', '0.03'
]

print(f"🚀 Comando: {' '.join(cmd)}\n")
result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n✅ Evaluation completata!")
else:
    print(f"\n❌ Errore (exit code: {result.returncode})")

🚀 Comando: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/bin/python experiments/evaluate.py --agent results/hierarchical_agent_3/hierarchical_agent.pt --victim-model checkpoints/victim/resnet18_pathmnist_best.pth --dataset pathmnist --n-samples 500 --output-dir results/evaluation --epsilon 0.03



2026-02-08 14:50:41,207 - __main__ - INFO - Native mode: 28x28 images
2026-02-08 14:50:41,207 - __main__ - INFO - Normalization stats: mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]
2026-02-08 14:50:41,493 - __main__ - INFO - Loading hierarchical agent from results/hierarchical_agent_3/hierarchical_agent.pt


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


2026-02-08 14:50:42,919 - __main__ - INFO - Hierarchical agent loaded successfully
2026-02-08 14:50:42,919 - __main__ - INFO -   Meta-Controller epsilon: 0.0
2026-02-08 14:50:42,919 - __main__ - INFO -   Timesteps trained: 0


Loading model from: /home/demetra/Desktop/Manuel/github-repo/MedSecure/.venv/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth


2026-02-08 14:50:43,231 - __main__ - INFO - Initialized Evaluator
2026-02-08 14:50:43,231 - __main__ - INFO -   Victim model: resnet18
2026-02-08 14:50:43,231 - __main__ - INFO -   Device: cuda
2026-02-08 14:50:43,231 - __main__ - INFO -   Test samples: 7180
2026-02-08 14:50:43,231 - __main__ - INFO - Evaluating hierarchical RL agent on 500 samples...
2026-02-08 14:50:43,231 - __main__ - INFO -   Epsilon budget: 0.03
2026-02-08 14:50:43,231 - __main__ - INFO -   Epsilon budget: 0.03
2026-02-08 14:50:43,231 - __main__ - INFO -   Environment epsilon_range after set: [0.005, 0.03]
2026-02-08 14:50:43,231 - __main__ - INFO - Allowed strategies: ['PIXEL', 'FREQUENCY', 'SEMANTIC']
Evaluating agent:   0%|          | 0/500 [00:00<?, ?it/s]2026-02-08 14:50:43,811 - __main__ - INFO - 
=== Sample 0 ===
2026-02-08 14:50:43,811 - __main__ - INFO -   Strategy: PIXEL
2026-02-08 14:50:43,811 - __main__ - INFO -   Steps: 5, Queries: 124
2026-02-08 14:50:43,811 - __main__ - INFO -   Epsilon budget: 0.03


EVALUATION SUMMARY
Method                 ASR       SSIM       PSNR    Queries
----------------------------------------------------------------------
MedSecure           16.87%     0.9926      48.81       67.3
FGSM                80.66%     0.9098      30.47        3.0
PGD                 91.99%     0.9430      32.49       43.0
BIM                 91.99%     0.9451      32.78       43.0

MedSecure Strategy Distribution:
  PIXEL       :    11 (  2.2%)
  FREQUENCY   :   295 ( 59.2%)
  SEMANTIC    :   192 ( 38.6%)


Results saved to results/evaluation/evaluation_results.json

✅ Evaluation completata!


In [64]:
import torch
ckpt = torch.load('checkpoints/victim/resnet18_pathmnist_best.pth', map_location='cpu')
print('native:', ckpt.get('native', 'NON PRESENTE'))
print('Keys:', list(ckpt.keys()))

native: True
Keys: ['epoch', 'model_name', 'dataset_name', 'n_classes', 'state_dict', 'accuracy', 'history', 'normalize_stats', 'native']


In [ ]:
from rl.hierarchical_agent import HierarchicalAgent

# Carica agente
ckpt = torch.load('PERCORSO_AGENTE/hierarchical_agent.pt', map_location='cpu')

# Simula un input
state = torch.randn(1, 23)  # 20 state + 3 strategies
policy_state = ckpt['controller_policy']

# Carica policy weights
from rl.hierarchical_agent import PolicyNetwork
policy = PolicyNetwork(state_dim=20, n_strategies=3, param_dim=8)
policy.load_state_dict(policy_state)

# Get action
with torch.no_grad():
    action, _, _ = policy.get_action(state, deterministic=True)
    
print('Policy output (deterministic):')
print(f'  params[0] (epsilon): {action[0, 0].item():.4f}')
print(f'  params[1] (iterations): {action[0, 1].item():.4f}')
print(f'  All params: {action[0].tolist()}')


Keys: ['meta_q_network', 'meta_target_network', 'meta_epsilon', 'controller_policy', 'total_timesteps', 'episode_count', 'config']
total_timesteps: 0
episode_count: 0
log_std: shape=torch.Size([8]), mean=0.0000, std=0.0000
features.0.weight: shape=torch.Size([128, 23]), mean=0.0015, std=0.1079
features.0.bias: shape=torch.Size([128]), mean=0.0154, std=0.1166
features.2.weight: shape=torch.Size([128, 128]), mean=-0.0090, std=0.0516
features.2.bias: shape=torch.Size([128]), mean=-0.0025, std=0.0505
mean.weight: shape=torch.Size([8, 128]), mean=-0.0001, std=0.0499
mean.bias: shape=torch.Size([8]), mean=-0.0161, std=0.0557
value.weight: shape=torch.Size([1, 128]), mean=0.0002, std=0.0205
value.bias: shape=torch.Size([1]), mean=0.0329, std=nan


/tmp/ipykernel_72190/248665193.py:11: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  print(f'{k}: shape={v.shape}, mean={v.mean().item():.4f}, std={v.std().item():.4f}')


In [46]:
ckpt = torch.load('results/hierarchical_agent_2/checkpoint_75000/hierarchical_agent.pt', map_location='cpu')
print('total_timesteps:', ckpt.get('total_timesteps', 'NON PRESENTE'))
print('episode_count:', ckpt.get('episode_count', 'NON PRESENTE'))
print('Keys:', list(ckpt.keys()))

total_timesteps: 0
episode_count: 0
Keys: ['meta_q_network', 'meta_target_network', 'meta_epsilon', 'controller_policy', 'total_timesteps', 'episode_count', 'config']


In [34]:

import torch
ckpt = torch.load('/home/demetra/Desktop/Manuel/github-repo/MedSecure/results/hierarchical_agent_2/hierarchical_agent.pt', map_location='cpu')
policy_state = ckpt['controller_policy']
# Trova la dimensione dell'input del primo layer
for k, v in policy_state.items():
    if 'weight' in k:
        print(f'{k}: {v.shape}')
        break


features.0.weight: torch.Size([128, 23])


In [35]:

import torch
ckpt = torch.load('/home/demetra/Desktop/Manuel/github-repo/MedSecure/results/hierarchical_agent_2/hierarchical_agent.pt', map_location='cpu')
policy_state = ckpt['controller_policy']
for k, v in policy_state.items():
    if 'mean' in k and 'weight' in k:
        print(f'{k}: {v.shape}')


mean.weight: torch.Size([8, 128])


In [ ]:
# Visualizza risultati finali
import json
from pathlib import Path

report_path = Path('results/evaluation/report.json')

if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)

    print("📊 FINAL RESULTS")
    print("="*50)
    print(f"Attack Success Rate: {report.get('asr', 0):.2%}")
    print(f"SSIM: {report.get('ssim', 0):.3f}")
    print(f"PSNR: {report.get('psnr', 0):.2f} dB")
    print(f"LPIPS: {report.get('lpips', 0):.3f}")
    print("="*50)
else:
    print(f"❌ Report non trovato: {report_path}")

## 9️⃣ Visualizzazione

In [ ]:
# Visualizza esempi adversarial
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

examples_dir = Path('results/evaluation/adversarial_examples')
example_files = list(examples_dir.glob('*.png'))[:6] if examples_dir.exists() else []

if example_files:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(example_files):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(f'Example {idx+1}')
    
    # Nascondi assi vuoti
    for idx in range(len(example_files), 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('results/evaluation/examples_grid.png', dpi=150)
    plt.show()
else:
    print("Nessun esempio trovato in", examples_dir)

## Riepilogo



### File generati:
- `checkpoints/victim/` - Modello vittima
- `results/baselines/` - Risultati attacchi baseline
- `results/agent/` - Agente RL addestrato
- `results/evaluation/` - Report e visualizzazioni